# Sleep-EDF Expanded | Single-Channel EEG | SHAP Feature Selection

## Import & Global Configurations

In [ ]:
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings("ignore")

# Signal processing
import scipy.signal as signal
from scipy.stats import skew, kurtosis

# Entropy
from antropy import perm_entropy, spectral_entropy

# ML
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    classification_report,
    confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier

# XGBoost
from xgboost import XGBClassifier

# SHAP
import shap

# Stats
from scipy.stats import wilcoxon

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## Dataset Loader (Sleep-EDF Expanded)

In [ ]:
import mne
import numpy as np
from pathlib import Path

BASE_PATH = "/home/agribychaniago/Python Projects/Sleep EDF/physionet.org/files/sleep-edfx/1.0.0"
CASSETTE_PATH = os.path.join(BASE_PATH, "sleep-cassette")
TELEMETRY_PATH = os.path.join(BASE_PATH, "sleep-telemetry")

def load_sleep_edf(subject_id, channel, dataset="cassette"):
    base_path = Path("physionet.org/files/sleep-edfx/1.0.0") / f"sleep-{dataset}"

    psg_path = base_path / f"{subject_id}-PSG.edf"
    hyp_candidates = list(base_path.glob(f"{subject_id[:-1]}*-Hypnogram.edf"))
    hyp_path = hyp_candidates[0]

    raw = mne.io.read_raw_edf(psg_path, preload=True, verbose=False)
    raw.pick(channel)

    annotations = mne.read_annotations(hyp_path)
    raw.set_annotations(annotations)

    # 1. Biarkan MNE mendeteksi SEMUA event
    events, event_id = mne.events_from_annotations(
        raw,
        chunk_duration=30.0
    )
    print(f"Event ID yang terdeteksi: {event_id}")

    # 2. DEFINISIKAN HANYA ANOTASI YANG INGIN KITA PAKAI
    # Ini adalah daftar "whitelist" kita
    wanted_stages = [
        "Sleep stage W",
        "Sleep stage 1",
        "Sleep stage 2",
        "Sleep stage 3",
        "Sleep stage 4",
        "Sleep stage R",
    ]

    # 3. FILTER events dan event_id berdasarkan "whitelist"
    final_event_id = {k: v for k, v in event_id.items() if k in wanted_stages}
    
    # Dapatkan daftar event ID yang diinginkan
    wanted_event_ids = list(final_event_id.values())
    
    # Filter array events, hanya simpan yang ID-nya ada di whitelist
    events = events[np.isin(events[:, 2], wanted_event_ids)]
    
    print(f"Event ID setelah filter: {final_event_id}")
    print(f"Jumlah epoch setelah filter: {len(events)}")

    # 4. Buat epochs dengan data yang SUDAH BERSIH
    epochs = mne.Epochs(
        raw,
        events,
        event_id=final_event_id,
        tmin=0,
        tmax=30,
        baseline=None,
        preload=True,
        verbose=False
    )

    X = epochs.get_data()[:, 0, :]   # (n_epochs, n_samples)
    
    # 5. Remap label (sekarang 100% aman karena tidak ada label tak terduga)
    label_map = {
        final_event_id["Sleep stage W"]: 0,
        final_event_id["Sleep stage 1"]: 1,
        final_event_id["Sleep stage 2"]: 2,
        final_event_id["Sleep stage 3"]: 3,
        final_event_id["Sleep stage 4"]: 3,  # merge S3+S4
        final_event_id["Sleep stage R"]: 4,
    }
    
    y_raw = epochs.events[:, -1]
    y = np.array([label_map[l] for l in y_raw]) # Tidak perlu .get() lagi
    
    # Verifikasi distribusi kelas
    unique, counts = np.unique(y, return_counts=True)
    print(f"Distribusi kelas AKHIR: {dict(zip(unique, counts))}")
    
    return X, y

## Feature Extraction (≈160–180 fitur)
* Time-domain
* Frequency-domain
* Hjorth
* Permutation Entropy
* Spectral Entropy

In [ ]:
from antropy import (
    perm_entropy,
    spectral_entropy,
    sample_entropy,
    app_entropy,
    higuchi_fd,
    petrosian_fd,
    lziv_complexity
)
import numpy as np
import scipy.signal as signal
from scipy.stats import skew, kurtosis


def extract_features(epoch, sfreq):
    features = {}

    # ======================
    # Time-domain statistics
    # ======================
    features["mean"] = np.mean(epoch)
    features["std"] = np.std(epoch)
    features["var"] = np.var(epoch)
    features["skew"] = skew(epoch, bias=False)
    features["kurtosis"] = kurtosis(epoch, bias=False)
    features["rms"] = np.sqrt(np.mean(epoch ** 2))
    features["ptp"] = np.ptp(epoch)
    features["median"] = np.median(epoch)
    features["iqr"] = np.percentile(epoch, 75) - np.percentile(epoch, 25)

    zero_crossings = np.where(np.diff(np.signbit(epoch)))[0]
    features["zero_crossing_rate"] = len(zero_crossings) / len(epoch)

    # ======================
    # Hjorth parameters
    # ======================
    diff1 = np.diff(epoch)
    diff2 = np.diff(diff1)

    var0 = np.var(epoch) + 1e-10
    var1 = np.var(diff1) + 1e-10
    var2 = np.var(diff2) + 1e-10

    features["hjorth_activity"] = var0
    features["hjorth_mobility"] = np.sqrt(var1 / var0)
    features["hjorth_complexity"] = np.sqrt(var2 / var1) / features["hjorth_mobility"]

    # ======================
    # Frequency-domain (Welch PSD)
    # ======================
    nperseg = int(4 * sfreq)
    freqs, psd = signal.welch(epoch, sfreq, nperseg=nperseg)
    total_power = np.trapz(psd, freqs) + 1e-10

    bands = {
        "delta": (0.5, 4),
        "theta": (4, 8),
        "alpha": (8, 13),
        "beta": (13, 30),
        "gamma": (30, 45)
    }

    band_powers = {}

    for band, (low, high) in bands.items():
        idx = (freqs >= low) & (freqs <= high)
        band_psd = psd[idx]
        band_freqs = freqs[idx]

        bp = np.trapz(band_psd, band_freqs)
        band_powers[band] = bp

        features[f"{band}_power"] = bp
        features[f"{band}_rel_power"] = bp / total_power
        features[f"{band}_log_power"] = np.log(bp + 1e-10)

        features[f"{band}_psd_mean"] = np.mean(band_psd)
        features[f"{band}_psd_std"] = np.std(band_psd)
        features[f"{band}_psd_max"] = np.max(band_psd)
        features[f"{band}_psd_skew"] = skew(band_psd, bias=False)
        features[f"{band}_psd_kurtosis"] = kurtosis(band_psd, bias=False)

    # ======================
    # Band ratios
    # ======================
    features["delta_theta_ratio"] = band_powers["delta"] / (band_powers["theta"] + 1e-10)
    features["theta_alpha_ratio"] = band_powers["theta"] / (band_powers["alpha"] + 1e-10)
    features["alpha_beta_ratio"] = band_powers["alpha"] / (band_powers["beta"] + 1e-10)

    # ======================
    # Global spectral descriptors
    # ======================
    features["spectral_centroid"] = np.sum(freqs * psd) / np.sum(psd)
    features["spectral_bandwidth"] = np.sqrt(
        np.sum(((freqs - features["spectral_centroid"]) ** 2) * psd) / np.sum(psd)
    )
    features["spectral_flatness"] = np.exp(np.mean(np.log(psd + 1e-10))) / np.mean(psd)

    cumulative_energy = np.cumsum(psd)
    features["spectral_rolloff_85"] = freqs[np.where(
        cumulative_energy >= 0.85 * cumulative_energy[-1]
    )[0][0]]

    # ======================
    # Entropy & complexity
    # ======================
    features["perm_entropy"] = perm_entropy(epoch, normalize=True)
    features["spectral_entropy"] = spectral_entropy(epoch, sfreq, normalize=True)
    features["sample_entropy"] = sample_entropy(epoch)
    features["approx_entropy"] = app_entropy(epoch)

    # ======================
    # Fractal & nonlinear complexity
    # ======================
    features["higuchi_fd"] = higuchi_fd(epoch)
    features["petrosian_fd"] = petrosian_fd(epoch)
    features["lziv_complexity"] = lziv_complexity(epoch > np.mean(epoch))

    return features


## Build Feature Matrix

In [ ]:
def build_feature_dataframe(X, sfreq, subject_id):
    rows = []
    for epoch in X:
        feats = extract_features(epoch, sfreq)
        feats["subject"] = subject_id
        rows.append(feats)
    return pd.DataFrame(rows)


## Full Dataset Assembly

In [ ]:
all_subjects = ["SC4001E0", "SC4002E0", "SC4011E0", "SC4012E0", "SC4021E0"]

df_list = []
y_list = []

for subject in all_subjects:
    print(f"\nProcessing {subject}...")
    X, y = load_sleep_edf(
        subject_id=subject,
        channel="EEG Fpz-Cz",
        dataset="cassette"
    )

    sfreq = 100  # Sleep-EDF sampling rate

    df_feat = build_feature_dataframe(X, sfreq, subject)

    # ============================
    # SAFETY CHECK (penting untuk sidang)
    # ============================
    assert len(df_feat) == len(y), f"Epoch-label mismatch for {subject}"

    df_list.append(df_feat)
    y_list.append(y)

# ============================
# Final dataset
# ============================
X_df = pd.concat(df_list, ignore_index=True)
y = np.concatenate(y_list)

groups = X_df["subject"].values
X_df = X_df.drop(columns=["subject"])

# ============================
# VERIFIKASI AKHIR (WAJIB)
# ============================
print("\n" + "="*50)
print("Verifikasi distribusi kelas AKHIR:")
print(np.unique(y, return_counts=True))
print(f"Total epochs: {len(y)}")
print(f"Total subjects: {len(all_subjects)}")
print("="*50)

Used Annotations descriptions: [np.str_('Sleep stage 1'), np.str_('Sleep stage 2'), np.str_('Sleep stage 3'), np.str_('Sleep stage 4'), np.str_('Sleep stage ?'), np.str_('Sleep stage R'), np.str_('Sleep stage W')]
Event ID yang terdeteksi: {np.str_('Sleep stage 1'): 1, np.str_('Sleep stage 2'): 2, np.str_('Sleep stage 3'): 3, np.str_('Sleep stage 4'): 4, np.str_('Sleep stage ?'): 5, np.str_('Sleep stage R'): 6, np.str_('Sleep stage W'): 7}
Event ID setelah filter: {np.str_('Sleep stage 1'): 1, np.str_('Sleep stage 2'): 2, np.str_('Sleep stage 3'): 3, np.str_('Sleep stage 4'): 4, np.str_('Sleep stage R'): 6, np.str_('Sleep stage W'): 7}
Jumlah epoch setelah filter: 2650
Distribusi kelas AKHIR: {np.int64(0): np.int64(1996), np.int64(1): np.int64(58), np.int64(2): np.int64(250), np.int64(3): np.int64(220), np.int64(4): np.int64(125)}
Used Annotations descriptions: [np.str_('Movement time'), np.str_('Sleep stage 1'), np.str_('Sleep stage 2'), np.str_('Sleep stage 3'), np.str_('Sleep stage 

### Verify Data Loaders

harus gini: (array([0, 1, 2, 3, 4]), counts)

In [ ]:
print(np.unique(y, return_counts=True))


(array([0, 1, 2, 3, 4]), array([3880,  117,  623,  517,  340]))


## Cross-Validation (SGK-Fold)

In [ ]:
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


## Models Definition

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=len(np.unique(y)),
    eval_metric="mlogloss",
    tree_method="hist",   # lebih cepat & stabil
    random_state=RANDOM_STATE,
    n_jobs=-1
)


## Experiment Loop (BASELINE vs SHAP)

In [ ]:
results = []

for fold, (train_idx, test_idx) in enumerate(sgkf.split(X_df, y, groups)):
    print(f"\n=== Fold {fold+1} ===")

    X_train, X_test = X_df.iloc[train_idx], X_df.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # ---- Scaling baseline ----
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    # ---- Baseline XGB ----
    xgb_base = XGBClassifier(**xgb_model.get_params())
    xgb_base.fit(X_train_s, y_train)
    y_pred_base = xgb_base.predict(X_test_s)

    f1_base = f1_score(y_test, y_pred_base, average="macro")

    # ---- SHAP Feature Selection ----
    explainer = shap.TreeExplainer(xgb_base)
    shap_vals = explainer.shap_values(X_train_s)

    # DEBUG: Cek bentuk array SHAP (sangat membantu)
    # print(f"Shape shap_vals: {np.array(shap_vals).shape}") # Contoh output: (n_samples, 67, 5)

    # PERBAIKAN: Rata-ratakan di axis sampel (0) dan kelas (2)
    shap_importance = np.mean(np.abs(shap_vals), axis=(0, 2))

    # Sekarang panjang shap_importance akan sama dengan jumlah fitur (67)
    feat_importance = pd.Series(shap_importance, index=X_df.columns)

    selected_feats = feat_importance.sort_values(
        ascending=False
    ).head(int(0.5 * len(feat_importance))).index.tolist()

    # ---- Scaling selected features ----
    scaler_sel = StandardScaler()
    X_train_sel = scaler_sel.fit_transform(X_train[selected_feats])
    X_test_sel = scaler_sel.transform(X_test[selected_feats])

    xgb_shap = XGBClassifier(**xgb_model.get_params())
    xgb_shap.fit(X_train_sel, y_train)
    y_pred_shap = xgb_shap.predict(X_test_sel)

    f1_shap = f1_score(y_test, y_pred_shap, average="macro")

    results.append([f1_base, f1_shap])



=== Fold 1 ===

=== Fold 2 ===

=== Fold 3 ===


ValueError: Found array with 0 sample(s) (shape=(0, 67)) while a minimum of 1 is required by StandardScaler.

## Statistical Test

In [ ]:
results = np.array(results)

stat, p = wilcoxon(results[:, 0], results[:, 1])

print("Baseline Macro F1:", results[:, 0].mean())
print("SHAP Macro F1:", results[:, 1].mean())
print("Wilcoxon p-value:", p)


## BoxPlot Performa Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

plt.figure(figsize=(6, 5))
sns.boxplot(
    data=pd.DataFrame({
        "Baseline XGB": results[:, 0],
        "SHAP-XGB": results[:, 1]
    })
)
plt.ylabel("Macro F1-score")
plt.title("Macro F1 Distribution Across SGK-Folds")
plt.show()


## Paired Line Plot

In [ ]:
plt.figure(figsize=(6, 5))

for i in range(len(results)):
    plt.plot(
        ["Baseline XGB", "SHAP-XGB"],
        results[i],
        marker="o",
        color="gray",
        alpha=0.6
    )

plt.ylabel("Macro F1-score")
plt.title("Paired Macro F1 per SGK-Fold")
plt.grid(True)
plt.show()


## Confusion Matrix Visualization (Aggregate)

In [ ]:
# --- Collect predictions across folds ---
y_true_all = []
y_pred_all = []

for train_idx, test_idx in sgkf.split(X_df, y, groups):

    X_train, X_test = X_df.iloc[train_idx], X_df.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    # Train model
    xgb_model.fit(X_train_s, y_train)

    # SHAP-based feature selection
    explainer = shap.TreeExplainer(xgb_model)
    shap_vals = explainer.shap_values(X_train_s)

    shap_importance = np.mean(np.abs(shap_vals), axis=(0, 1))
    selected_feats = pd.Series(
        shap_importance, index=X_df.columns
    ).sort_values(ascending=False).head(
        int(0.5 * X_df.shape[1])
    ).index

    # Retrain with selected features
    X_train_sel = scaler.fit_transform(X_train[selected_feats])
    X_test_sel = scaler.transform(X_test[selected_feats])

    xgb_model.fit(X_train_sel, y_train)
    y_pred = xgb_model.predict(X_test_sel)

    y_true_all.extend(y_test)
    y_pred_all.extend(y_pred)


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

class_names = ["W", "N1", "N2", "N3", "REM"]

cm = confusion_matrix(y_true_all, y_pred_all)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

plt.figure(figsize=(6, 6))
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix (SHAP-XGB, Aggregated SGK-Folds)")
plt.tight_layout()
plt.show()


## SHAP Feature Importance Visualization (Top 20)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot style
sns.set(style="whitegrid")

# Select top-N features
top_n = 20
top_features = (
    feat_importance
    .sort_values(ascending=False)
    .head(top_n)
)

# Plot
plt.figure(figsize=(7, 6))
sns.barplot(
    x=top_features.values,
    y=top_features.index,
    orient="h"
)

plt.xlabel("Mean |SHAP value|")
plt.ylabel("Feature")
plt.title("Top-20 SHAP Feature Importance (Representative Fold)")
plt.tight_layout()
plt.show()


## Feature Stability Plot

In [ ]:
from collections import Counter

feature_counter = Counter()

# Simpan selected_feats tiap fold (modifikasi loop sebelumnya)
# feature_counter.update(selected_feats)

most_common = feature_counter.most_common(20)
features, counts = zip(*most_common)

plt.figure(figsize=(7, 6))
sns.barplot(x=counts, y=features)
plt.xlabel("Selection Frequency Across Folds")
plt.title("Feature Stability Across SGK-Folds")
plt.show()


## Correlation Heatmap Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot style
sns.set(style="white")

# Hitung korelasi (subset 30x30 pertama)
corr_subset = X_df.corr().iloc[:30, :30]

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_subset,
    cmap="coolwarm",
    center=0,
    square=True,
    cbar_kws={"shrink": 0.8}
)

plt.title("Feature Correlation Heatmap (Top-Left 30×30 Subset)")
plt.tight_layout()
plt.show()
